<a href="https://colab.research.google.com/github/BenMillerDev/Applied-LLM-Systems/blob/week-4-tool-use/week-4/week4_tool_use.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4: Multi-Tool Assistant -- PR Triage
### Ben Miller

Domain: a software repo / code review assistant. This is similar to a tool my team at work uses to classify pull requests
into size tiers (low/medium/high).  For this exercise I use a mock repo with file names and number of lines changed, and the tools use it to determine how to classify a PR.

The three tools:
- `get_file_stats(path)` -- look up how many lines of a file the pending PR touches.
- `classify_pr_size(lines_changed, complexity)` -- turn a line count and a
  complexity label into a low/medium/high tier.
- `run_python(expression)` -- a guarded code-runner, used to sum `lines_changed`
  across files when a PR touches more than one.

A PR with one file goes `get_file_stats` -> `classify_pr_size` (two tools). A PR
with several files goes `get_file_stats` (once per file) -> `run_python` (sum) ->
`classify_pr_size` (three tools, two of them chained on a value the model has to
carry forward)

In [1]:
import os, ast, operator, json, math, signal

try:
    from google.colab import userdata
    _key = userdata.get('GEMINI_API_KEY')
    if _key:
        os.environ['GEMINI_API_KEY'] = _key
except Exception:
    pass  # not running in Colab, or the secret isn't set there -- fall back to whatever's already in the environment

HAS_API_KEY = bool(os.environ.get('GEMINI_API_KEY', '').strip())
print('GEMINI_API_KEY set:', HAS_API_KEY)
# The scripted sanity-check loop below runs either way regardless of HAS_API_KEY.

GEMINI_API_KEY set: True


## Part 1: Tool schemas

`path` on `get_file_stats` is deliberately left as an open string rather than an
enum.  The set of files in a repo isn't a fixed, small vocabulary the way units
or complexity levels are, so an unrecognized path is a genuine runtime failure
(`execution_error`), not a bad-argument one. `complexity` is an enum because it's
a closed judgment call with exactly three values my team actually uses.

In [2]:
TOOLS = [
  {'name':'get_file_stats','description':'Look up how many lines of a file are touched by the pending pull request.',
   'parameters':{'type':'object','properties':{'path':{'type':'string'}},'required':['path']}},
  {'name':'classify_pr_size','description':'Classify a pull request into a low/medium/high size tier from its total lines changed and how mechanical vs. substantial the change is.',
   'parameters':{'type':'object','properties':{
       'lines_changed':{'type':'number'},
       'complexity':{'type':'string','enum':['mechanical','moderate','substantial']}},
     'required':['lines_changed','complexity']}},
  {'name':'run_python','description':'Run a single allowlisted arithmetic expression, e.g. to sum lines_changed across multiple files (guarded code-runner).',
   'parameters':{'type':'object','properties':{'expression':{'type':'string'}},'required':['expression']}},
]
print('tools:', [t['name'] for t in TOOLS])

tools: ['get_file_stats', 'classify_pr_size', 'run_python']


## Mock repo data

Stands in for a real diff/PR API. `lines_changed` is how many lines of that
file the *pending PR* touches, not the file's total size.

In [3]:
FAKE_REPO = {
  'src/api/routes.py':              {'lines_changed': 42},
  'src/api/handlers.py':            {'lines_changed': 187},
  'src/models/user.py':             {'lines_changed': 15},
  'migrations/0042_add_index.sql':  {'lines_changed': 6},
  'vendor/generated_client.py':     {'lines_changed': 512},
  'tests/test_handlers.py':         {'lines_changed': 64},
}
print('files in fake repo:', list(FAKE_REPO))

files in fake repo: ['src/api/routes.py', 'src/api/handlers.py', 'src/models/user.py', 'migrations/0042_add_index.sql', 'vendor/generated_client.py', 'tests/test_handlers.py']


## Part 2: Guarded code-runner (`run_python`)

**Allowlist, checked before execution.** `run_python` parses the expression into
a Python AST and walks it with `_safe()`. It only recognizes three node types:
numeric constants, binary arithmetic operators (`+ - * / **`), and unary negation.  Anything else raises a `ValueError`.

This blocks three categories of risk:
- **Filesystem access.** No `open` or `Path`, since names never resolve to anything.
- **Network access.** No `socket`, `requests`, or `urllib`, since no calls can be
  reached.
- **Process execution.** No `os.system`, `subprocess`, or `__import__`, since both
  imports and attribute access are unrecognized node types.

**Time limit.** The allowlist alone doesn't stop resource exhaustion.  `run_python` wraps evaluation in `signal.alarm(timeout_seconds)` (2 seconds by
default) and raises `ExpressionTimeout` if it fires, always cancelling the alarm
in a `finally` block afterward.


In [4]:
_OPS = {ast.Add:operator.add, ast.Sub:operator.sub, ast.Mult:operator.mul, ast.Div:operator.truediv, ast.Pow:operator.pow, ast.USub:operator.neg}
def _safe(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int,float): return node.value
    if isinstance(node, ast.BinOp): return _OPS[type(node.op)](_safe(node.left), _safe(node.right))
    if isinstance(node, ast.UnaryOp): return _OPS[type(node.op)](_safe(node.operand))
    raise ValueError('only arithmetic is allowed')   # blocks names, calls, imports, attributes

class ExpressionTimeout(Exception): pass
def _raise_timeout(signum, frame):
    raise ExpressionTimeout('expression took too long to evaluate')

def run_python(expression, timeout_seconds=2):
    signal.signal(signal.SIGALRM, _raise_timeout)
    signal.alarm(timeout_seconds)
    try:
        return _safe(ast.parse(expression, mode='eval').body)
    finally:
        signal.alarm(0)  # cancel so a stale alarm can't fire on some later, unrelated call

def get_file_stats(path):
    if path not in FAKE_REPO:
        raise ValueError(f'no such file in this PR: {path}')
    return {'path': path, **FAKE_REPO[path]}

def classify_pr_size(lines_changed, complexity):
    if lines_changed < 50: base = 'low'
    elif lines_changed <= 300: base = 'medium'
    else: base = 'high'
    tiers = ['low', 'medium', 'high']
    idx = tiers.index(base)
    if complexity == 'mechanical': idx = max(0, idx - 1)      # large but boilerplate reviews easier than the raw count suggests
    elif complexity == 'substantial': idx = min(2, idx + 1)   # small but gnarly reviews harder than the raw count suggests
    return tiers[idx]

IMPL = {'get_file_stats':get_file_stats, 'classify_pr_size':classify_pr_size, 'run_python':run_python}

`ToolArgError`, `validate()`, and `dispatch()` are from the provided starter code

In [5]:
class ToolArgError(Exception): pass
def validate(name, args):
    spec = next((t['parameters'] for t in TOOLS if t['name']==name), None)
    if spec is None: raise ToolArgError(f'unknown tool: {name}')
    if not isinstance(args, dict): raise ToolArgError('arguments must be a JSON object')
    for r in spec.get('required',[]):
        if r not in args: raise ToolArgError(f'missing required field: {r}')
    for k,v in args.items():
        p = spec['properties'].get(k)
        if p is None: raise ToolArgError(f'unexpected field: {k}')
        if p['type'] == 'string':
            if not isinstance(v, str): raise ToolArgError(f'{k} must be a string')
        elif p['type'] == 'number':
            if type(v) not in (int, float): raise ToolArgError(f'{k} must be a number (not a boolean)')
            if isinstance(v, float) and not math.isfinite(v): raise ToolArgError(f'{k} must be finite')
        else:
            raise ValueError(f'extend validate() to support schema type: {p["type"]}')
        if 'enum' in p and v not in p['enum']: raise ToolArgError(f'{k}={v!r} not in {p["enum"]}')
def dispatch(name, args):
    try:
        validate(name, args)
        output = IMPL[name](**args)
        json.dumps(output, allow_nan=False)  # reject results the model cannot receive as JSON
        return {'ok':True,'tool':name,'output':output}
    except ToolArgError as e:
        return {'ok':False,'tool':name,'error_type':'invalid_arguments','message':str(e)}
    except Exception as e:
        return {'ok':False,'tool':name,'error_type':'execution_error','message':str(e)}

### Sanity check

Scripted calls (no model involved yet) to confirm `dispatch` behaves before
wiring it to a live model: a single-file PR (two-tool chain), a multi-file PR
(three-tool chain via `run_python` for the sum), an invalid `complexity` enum
followed by a corrected retry, and the guarded-failure case for `run_python`.

In [6]:
scripted = [
  ('get_file_stats',   {'path':'src/models/user.py'}),
  ('classify_pr_size', {'lines_changed':15,'complexity':'moderate'}),
  ('get_file_stats',   {'path':'src/api/handlers.py'}),
  ('get_file_stats',   {'path':'tests/test_handlers.py'}),
  ('run_python',       {'expression':'187+64'}),
  ('classify_pr_size', {'lines_changed':251,'complexity':'mechanical'}),
  ('classify_pr_size', {'lines_changed':251,'complexity':'urgent'}),   # scripted invalid enum
  ('classify_pr_size', {'lines_changed':251,'complexity':'substantial'}),  # scripted corrected retry
  ('run_python',       {'expression':'__import__("os").system("echo hi")'}),  # guarded
]
for name, args in scripted:
    r = dispatch(name, args)
    tag = 'OK ' if r['ok'] else 'ERR'
    print(f'[{tag}] {name}({args}) -> {json.dumps(r, allow_nan=False)}')

[OK ] get_file_stats({'path': 'src/models/user.py'}) -> {"ok": true, "tool": "get_file_stats", "output": {"path": "src/models/user.py", "lines_changed": 15}}
[OK ] classify_pr_size({'lines_changed': 15, 'complexity': 'moderate'}) -> {"ok": true, "tool": "classify_pr_size", "output": "low"}
[OK ] get_file_stats({'path': 'src/api/handlers.py'}) -> {"ok": true, "tool": "get_file_stats", "output": {"path": "src/api/handlers.py", "lines_changed": 187}}
[OK ] get_file_stats({'path': 'tests/test_handlers.py'}) -> {"ok": true, "tool": "get_file_stats", "output": {"path": "tests/test_handlers.py", "lines_changed": 64}}
[OK ] run_python({'expression': '187+64'}) -> {"ok": true, "tool": "run_python", "output": 251}
[OK ] classify_pr_size({'lines_changed': 251, 'complexity': 'mechanical'}) -> {"ok": true, "tool": "classify_pr_size", "output": "low"}
[ERR] classify_pr_size({'lines_changed': 251, 'complexity': 'urgent'}) -> {"ok": false, "tool": "classify_pr_size", "error_type": "invalid_arguments",

## Part 1 (remaining half): the live model loop
Uses Gemini's native tool calling through `google-genai`.  
`automatic_function_calling` is off, so Gemini only proposes
a call.  Each call comes with an `id`.  The result comes back tagged with the same `id`, which is how Gemini matches a result to the call that produced it.

`run_agent` is the turn-by-turn loop: ask,
run any proposed calls, repeat until there's a final answer or `max_turns` runs out.

In [7]:
!pip install -q google-genai
from google import genai
from google.genai import types

MODEL = 'gemini-3.1-flash-lite'

gemini_client = genai.Client(api_key=os.environ['GEMINI_API_KEY']) if HAS_API_KEY else None
gemini_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(name=t['name'], description=t['description'], parameters=t['parameters'])
    for t in TOOLS
])
print('gemini tool built from', len(gemini_tool.function_declarations), 'schemas; client ready:', gemini_client is not None)

gemini tool built from 3 schemas; client ready: True


In [8]:
def _ask_model(contents):
    # Send the conversation so far to Gemini and return its response for
    # this turn. Tool results already appended to `contents` are just more
    # turns in the same history -- Gemini doesn't need them called out
    # specially.
    return gemini_client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            tools=[gemini_tool],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        ),
    )


def _requested_calls(candidate):
    # The tool calls Gemini proposed this turn, if any
    # empty once it's ready to give a final answer instead."
    return [part.function_call for part in candidate.content.parts if part.function_call]


def _run_requested_call(call, turn, log, verbose):
    # Execute one proposed call through dispatch(), record it in `log`,
    # and return it as a FunctionResponse Part carrying the same call id Gemini
    # sent -- that id is what lets Gemini match this result back to its call."""
    args = call.args or {}
    result = dispatch(call.name, args)
    log.append({'turn': turn, 'call_id': call.id, 'name': call.name, 'args': args, 'result': result})
    if verbose:
        tag = 'OK ' if result['ok'] else 'ERR'
        print(f'[turn {turn}] [{tag}] {call.name}({args}) -> {result}')
    return types.Part(function_response=types.FunctionResponse(id=call.id, name=call.name, response=result))


def run_agent(query, max_turns=6, verbose=True):
    # Send `query` to Gemini with TOOLS attached, intercept every tool
    # call it makes, run each one through dispatch(), and feed the results
    # back -- tagged with the model's own call ids -- until it gives a final
    # text answer (or max_turns is hit).
    contents = [types.Content(role='user', parts=[types.Part(text=query)])]
    log = []

    for turn in range(max_turns):
        response = _ask_model(contents)
        candidate = response.candidates[0]
        contents.append(candidate.content)  # the model's own turn, calls included

        calls = _requested_calls(candidate)
        if not calls:
            final_text = response.text
            log.append({'turn': turn, 'final_text': final_text})
            if verbose: print(f'[turn {turn}] final answer: {final_text}')
            return final_text, log

        tool_response_parts = [_run_requested_call(call, turn, log, verbose) for call in calls]
        contents.append(types.Content(role='user', parts=tool_response_parts))

    final_text = f'[gave up: no final answer after {max_turns} turns]'
    log.append({'turn': max_turns, 'gave_up': True, 'final_text': final_text})
    if verbose: print(f'[gave up] no final answer after {max_turns} turns')
    return final_text, log

if HAS_API_KEY:
    answer, call_log = run_agent('How big is the PR touching src/models/user.py? It is a moderate-complexity change.')
    print()
    print('answer:', answer)
else:
    print('Set GEMINI_API_KEY as a Colab secret to run run_agent() live -- Part 3 needs at least three real queries like the one above.')

[turn 0] [OK ] get_file_stats({'path': 'src/models/user.py'}) -> {'ok': True, 'tool': 'get_file_stats', 'output': {'path': 'src/models/user.py', 'lines_changed': 15}}
[turn 1] [OK ] classify_pr_size({'complexity': 'moderate', 'lines_changed': 15}) -> {'ok': True, 'tool': 'classify_pr_size', 'output': 'low'}
[turn 2] final answer: The pull request touching `src/models/user.py` is classified as **low** size, given that it involves 15 lines of changes and is of moderate complexity.

answer: The pull request touching `src/models/user.py` is classified as **low** size, given that it involves 15 lines of changes and is of moderate complexity.


## Part 3: Three real queries
Query 1 is the two-tool chain (`get_file_stats` -> `classify_pr_size`). Query 2
needs `run_python` too, chaining all three tools. Query 3 shows `classify_pr_size`
shifting the tier down for a `mechanical` change.

In [9]:
queries = [
    "How big is the PR touching src/models/user.py? It's a moderate-complexity change.",
    "This PR changes src/api/handlers.py and tests/test_handlers.py. It's a substantial-complexity change overall -- what size tier is it?",
    "The PR only touches vendor/generated_client.py, and it's entirely auto-generated, mechanical boilerplate. What size tier is it?",
]

part3_runs = []
if HAS_API_KEY:
    for query in queries:
        print('QUERY:', query)
        answer, call_log = run_agent(query)
        part3_runs.append({'query': query, 'answer': answer, 'call_log': call_log})
        print()
        print('-' * 80)
        print()
else:
    print('Set GEMINI_API_KEY as a Colab secret to run these three queries live.')

QUERY: How big is the PR touching src/models/user.py? It's a moderate-complexity change.
[turn 0] [OK ] get_file_stats({'path': 'src/models/user.py'}) -> {'ok': True, 'tool': 'get_file_stats', 'output': {'path': 'src/models/user.py', 'lines_changed': 15}}
[turn 1] [OK ] classify_pr_size({'lines_changed': 15, 'complexity': 'moderate'}) -> {'ok': True, 'tool': 'classify_pr_size', 'output': 'low'}
[turn 2] final answer: The PR touching `src/models/user.py` has 15 lines changed and is classified as a **low** size tier, given its moderate complexity.

--------------------------------------------------------------------------------

QUERY: This PR changes src/api/handlers.py and tests/test_handlers.py. It's a substantial-complexity change overall -- what size tier is it?
[turn 0] [OK ] get_file_stats({'path': 'src/api/handlers.py'}) -> {'ok': True, 'tool': 'get_file_stats', 'output': {'path': 'src/api/handlers.py', 'lines_changed': 187}}
[turn 0] [OK ] get_file_stats({'path': 'tests/test_han

### Results Analysis
All three results match `classify_pr_size`'s rules.
- Query 1: 15 lines,
moderate -- base tier is low (under 50), moderate doesn't shift it, so low is
correct.
- Query 2: `run_python` correctly summed 187 + 64 = 251 across two files,
substantial complexity shifts the base tier (medium) up one to high -> correct.
- Query 3: the one case here where the tier-shift logic changes the answer.  512 lines would normally be high (over 300), but mechanical shifts it
down one to medium -> correct

Tool use also matches what each query needed: query 1 used the two-tool chain
(`get_file_stats` -> `classify_pr_size`), query 2 used all three (two parallel
`get_file_stats` calls, `run_python` to sum, then `classify_pr_size`), and every
call log above shows `[OK]`.  No invalid arguments or execution errors.

## Part 4: a failure and its recovery

`path` on `get_file_stats` is the one open-string field in these schemas, so it's
the one place a bad guess becomes a real failure instead of a rejected argument.
These three queries describe a file instead of naming it, forcing the model to guess.

In [10]:
probe_queries = [
    "How big is the change to the routes file? It's a moderate-complexity change.",
    "What size tier is the change to the file that adds the new database index? It's a mechanical change.",
    "This PR touches the user model and its handler file, both moderate complexity. What tier is it overall?",
]

part4_runs = []
if HAS_API_KEY:
    for query in probe_queries:
        print('QUERY:', query)
        answer, call_log = run_agent(query)
        part4_runs.append({'query': query, 'answer': answer, 'call_log': call_log})
        print()
        print('-' * 80)
        print()
else:
    print('Set GEMINI_API_KEY as a Colab secret to try these probes live.')

QUERY: How big is the change to the routes file? It's a moderate-complexity change.
[turn 0] [ERR] get_file_stats({'path': 'config/routes.rb'}) -> {'ok': False, 'tool': 'get_file_stats', 'error_type': 'execution_error', 'message': 'no such file in this PR: config/routes.rb'}
[turn 1] [ERR] run_python({'expression': '"config/routes.rb" in ["app/controllers/users_controller.rb", "app/models/user.rb", "config/routes.rb"]'}) -> {'ok': False, 'tool': 'run_python', 'error_type': 'execution_error', 'message': 'only arithmetic is allowed'}
[turn 2] [ERR] get_file_stats({'path': 'config/routes.rb'}) -> {'ok': False, 'tool': 'get_file_stats', 'error_type': 'execution_error', 'message': 'no such file in this PR: config/routes.rb'}
[turn 3] [ERR] get_file_stats({'path': 'routes.rb'}) -> {'ok': False, 'tool': 'get_file_stats', 'error_type': 'execution_error', 'message': 'no such file in this PR: routes.rb'}
[turn 4] [ERR] get_file_stats({'path': 'config/routes.rb'}) -> {'ok': False, 'tool': 'get_fi

## Failure results
All three guesses failed. `get_file_stats`'s `path` parameter is a plain string
with no enum, so nothing stops a wrong guess from reaching `dispatch`.

Clearest failure case: Probe 1 asked about "the routes file," and the model called
`get_file_stats({'path': 'config/routes.rb'})` -- a Rails-style guess for a Python
repo, which is not a key in `FAKE_REPO`. `dispatch` returned
`execution_error: no such file in this PR: config/routes.rb`. It repeated that
guess three times, tried two others, and gave up after 6 turns. The error says a
guess is wrong but never says what's right, so retrying blind never converged.

Probe 2 attempted to use `ls -R` to find the file path but was correctly blocked by the guard since that is not an allowed operation.

Probe 3 failed differently -- it invented line counts instead of running out of
turns. It guessed 200 lines.  The real value is 187, so the guess is very close.  However,  guesses like this could cause issues if they are near a threshold that would cause the final answer to change.  This is a separate issue from the one my fix addresses.

### The fix: more details in the tool description

I changed the tool description instead of the schema.  `path` stays a string.  A real repo's file list is too big for an enum, but this
mock PR's files are a small, fixed set, so the description now lists them directly.


In [11]:
TOOLS[0]['description'] = (
    'Look up how many lines of a file are touched by the pending pull request. '
    'Files touched by this PR: ' + ', '.join(sorted(FAKE_REPO)) + '.'
)
gemini_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(name=t['name'], description=t['description'], parameters=t['parameters'])
    for t in TOOLS
])
print(TOOLS[0]['description'])

Look up how many lines of a file are touched by the pending pull request. Files touched by this PR: migrations/0042_add_index.sql, src/api/handlers.py, src/api/routes.py, src/models/user.py, tests/test_handlers.py, vendor/generated_client.py.


### Test the recovery

In [12]:
if HAS_API_KEY:
    print('QUERY (retry after description fix):', probe_queries[0])
    answer, call_log = run_agent(probe_queries[0])
    print()
    print('answer:', answer)
else:
    print('Set GEMINI_API_KEY as a Colab secret to confirm the fix live.')

QUERY (retry after description fix): How big is the change to the routes file? It's a moderate-complexity change.
[turn 0] [OK ] get_file_stats({'path': 'src/api/routes.py'}) -> {'ok': True, 'tool': 'get_file_stats', 'output': {'path': 'src/api/routes.py', 'lines_changed': 42}}
[turn 1] [OK ] classify_pr_size({'complexity': 'moderate', 'lines_changed': 42}) -> {'ok': True, 'tool': 'classify_pr_size', 'output': 'low'}
[turn 2] final answer: The change to `src/api/routes.py` involves 42 lines. Given that this is a moderate-complexity change, it is classified as a **low-size** change.

answer: The change to `src/api/routes.py` involves 42 lines. Given that this is a moderate-complexity change, it is classified as a **low-size** change.


## Part 4: Fixed results
The same query now resolved in two calls with no wrong guesses: `get_file_stats`
found `src/api/routes.py` directly, `classify_pr_size` returned `low`. The file paths in the tool description allowed the model to find the correct files even when it was not told the path in the user's prompt.

## Part 5: Submit
Open a pull request with a schema design write-up, a link to this notebook, and
a link to an issue documenting the failure and recovery. Describe the
code-runner's allowlist, time limit, and blocked operations. Rubric: schemas
(20), loop including a two-step sequence (25), guarded code-runner (20),
failure with recovery (20), PR hygiene (15).